In [125]:
from pathlib import Path
import requests

import lxml
import pandas as pd
import numpy as np
import io

# get species table from website:

url = "https://www.pwrc.usgs.gov/BBL/Bander_Portal/login/speclist.php"
response = requests.get(url)

all_tables = pd.read_html(io.StringIO(response.text))
species_df = all_tables[0]
species_df.head()


,Species Number,Alpha Code,Common Name,Band Size,French Name,Scientific Name,T & E*,Comments,Taxonomic Order
0,10,WEGR,Western Grebe,"7B, 7, 7A",Grèbe élégant,Aechmophorus occidentalis,NaN,NaN,100
1,11,CLGR,Clark's Grebe,"7A, 7, 7B",Grèbe à face blanche,Aechmophorus clarkii,NaN,NaN,102
2,12,WCGR,Western/Clark's Grebe,"7A, 7B",Grèbe à face blanche/Grèbe élégant non identifiée,NaN,NaN,Unspecified Aechmophorus occidentalis or A. cl...,101
3,20,RNGR,Red-necked Grebe,"7A, 7, 7B",Grèbe jougris,Podiceps grisegena,NaN,NaN,98
4,30,HOGR,Horned Grebe,"6, 5, 5A",Grèbe esclavon,Podiceps auritus,Québec,Magdalen Islands population is endangered. | L...,97


In [148]:

def create_ebird_df():
    routes_df = pd.read_csv("data/routes.csv")
    trips_df = pd.read_csv("data/trips.csv")
    trips_df["id"] = trips_df.apply(lambda x: f"{int(x["route_id"])}-{int(x["yyyy"])}", axis=1)
    sites_df = pd.read_csv("data/sites.csv")
    sites_df["id"] = sites_df.apply(lambda x: f"{int(x["route_id"])}-{int(x["stop_number"])}", axis=1)

    ebird_data = {
            "Common Name": [],
            "Genus": [],
            "Species": [],
            "Number": [],
            "Species Comments": [],
            "Location Name": [],
            "Latitude": [],
            "Longitude": [],
            "Date": [],
            "Start Time": [],
            "State/Province": [],
            "Country Code": [],
            "Protocol": [],
            "Number of Observers": [],
            "Duration": [],
            "All observations reported?": [],
            "Effort Distance Miles": [],
            "Effort area acres": [],
            "Submission Comments": [],
        }

    for path1 in Path("data/raw_counts").iterdir():
        route = str(path1).split("/")[-1]
        for path2 in path1.iterdir():
            year = int(str(path2).split("/")[-1].replace(".csv", ""))
            trip_id = f"{route}-{year}"
            raw_data = pd.read_csv(path2, names=[f"col{x}" for x in range(1, 8)])

            # convert data into a better format
            data = dict(species=[], stop_number=[], count=[])

            recording = False
            sites = None
            for index, row in raw_data.iterrows():
                # if the first item in row is "Species" we are starting a new block.
                if row.iloc[0] == "Number of Vehicles":
                    recording = False
                elif row.iloc[0] == "Species":
                    recording = True
                    sites = [int(x) for x in row.iloc[1:6]]
                    continue
                if recording and sites:
                    # each col should be an observation in the new table
                    species = row.iloc[0]
                    tallies = row.iloc[1:6]
                    for t, site in zip(tallies, sites):
                        if pd.notna(t):
                            data["species"].append(species)
                            data["stop_number"].append(site)
                            data["count"].append(t)

            df = pd.DataFrame(data)
            df["route_id"] = target_route
            df["trip_id"] = trip_id

            # merge route information based on the route id
            df = df.merge(routes_df, left_on="route_id", right_on="id")
            # merge site information based on the trip id
            df["site_id"] = df.apply(lambda x: f"{int(x["route_id"])}-{int(x["stop_number"])}", axis=1)
            df = df.merge(sites_df, left_on="site_id", right_on="id")
            # merge trip information based on the trip id
            df = df.merge(trips_df, left_on="trip_id", right_on="id")

            for index, row in df.iterrows():
                ebird_data["Common Name"].append(row["species"])
                ebird_data["Genus"].append("")
                ebird_data["Species"].append("")
                ebird_data["Number"].append(int(row["count"]))
                ebird_data["Species Comments"].append(np.nan)
                ebird_data["Location Name"].append(f"BBS {row["name"]} ({row["route_id"]}) - Site {row["stop_number_x"]}" )
                ebird_data["Latitude"].append(row["latitude"])
                ebird_data["Longitude"].append(row["longitude"])
                ebird_data["Date"].append(f"{str(row["mm"]).rjust(2, "0")}/{str(row["dd"]).rjust(2, "0")}/{row["yyyy"]}")
                ebird_data["Start Time"].append(f"{str(row["start_hh"]).rjust(2, "0")}:{str(row["start_mm"]).rjust(2, "0")}")
                ebird_data["State/Province"].append(row["province"])
                ebird_data["Country Code"].append(row["country"])
                ebird_data["Protocol"].append("area")
                ebird_data["Number of Observers"].append(row["no_observers"])
                ebird_data["Duration"].append(3)
                ebird_data["All observations reported?"].append("Y")
                ebird_data["Effort Distance Miles"].append(np.nan)
                ebird_data["Effort area acres"].append(124)  # 400m radius
                ebird_data["Submission Comments"].append("Data submitted to USGS")

    ebird_df = pd.DataFrame(ebird_data)

    # corrections
    ebird_df.loc[ebird_df["Common Name"] == "(Myrtle Warbler) Yellow-rumped Warbler", "Common Name"] = "Myrtle Warbler"
    ebird_df.loc[ebird_df["Common Name"] == "(Slate-colored Junco) Dark-eyed Junco", "Common Name"] = "Slate-colored Junco"
    ebird_df.loc[ebird_df["Common Name"] == "(Yellow-shafted Flicker) Northern Flicker", "Common Name"] = "Yellow-shafted Flicker"
    ebird_df = ebird_df.loc[ebird_df["Number"] > 0]
    ebird_df.to_csv(f"ebird_output/bbs-checklist.csv", header=False, index=False)
    return ebird_df

df = create_ebird_df()


In [146]:
df.columns

Index(['Common Name', 'Genus', 'Species', 'Number', 'Species Comments',
       'Location Name', 'Latitude', 'Longitude', 'Date', 'Start Time',
       'State/Province', 'Country Code', 'Protocol', 'Number of Observers',
       'Duration', 'All observations reported?', 'Effort Distance Miles',
       'Effort area acres', 'Submission Comments'],
      dtype='str')

In [162]:
# unique bird list
df = df.loc[:, ["Common Name"]].drop_duplicates().sort_values(by=["Common Name"]).reset_index().drop(columns=["index"])

# a few additions to species codes
species_df.loc[len(species_df)] = {'Common Name': "Ring-necked Pheasant", 'Alpha Code': "RNEP"}
species_df.loc[len(species_df)] = {'Common Name': "Rock Pigeon", 'Alpha Code': "ROPI"}
species_df.loc[len(species_df)] = {'Common Name': "Ruffed Grouse", 'Alpha Code': "RUGR"}
species_df.loc[len(species_df)] = {'Common Name': "Wild Turkey", 'Alpha Code': "WITU"}

# join to species df
df_merged = df.merge(species_df, left_on="Common Name", right_on="Common Name", how="left")
df_merged.loc[:, ["Common Name", "Alpha Code"]].to_csv(f"ebird_output/unique-species-list.csv", header=True, index=False)
df_merged.head()

,Common Name,Species Number,Alpha Code,Band Size,French Name,Scientific Name,T & E*,Comments,Taxonomic Order
0,Alder Flycatcher,4661.0,ALFL,"0A, 0",Moucherolle des aulnes,Empidonax alnorum,NaN,AOU split from 4669-TRFL in 1973. | AOU l’a sé...,671.0
1,American Bittern,1900.0,AMBI,M: 7A F: 6,Butor d'Amérique,Botaurus lentiginosus,NaN,NaN,216.0
2,American Black Duck,1330.0,ABDU,7A,Canard noir,Anas rubripes,NaN,NaN,43.0
3,American Crow,4880.0,AMCR,"5, 5A",Corneille d'Amérique,Corvus brachyrhynchos,NaN,NaN,759.0
4,American Goldfinch,5290.0,AMGO,"0, 0A, 1C, 1",Chardonneret jaune,Spinus tristis,NaN,NaN,1204.0
